# Geomar AI Oxygen - Hypoxia Prediction Training (Google Colab)

Weighted hypoxia prediction model training pipeline on Google Colab.

**Requirements:**
- GPU runtime (Runtime > Change runtime type > GPU)
- Google Drive for checkpoints (recommended)
- Standard RAM sufficient

## 1. Setup Environment

In [ ]:
import torch
import pandas as pd
import numpy as np

print("="*80)
print("SYSTEM INFO")
print("="*80)
print(f"\nPyTorch: {torch.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem_gb:.1f} GB")

    if "A100" in gpu_name:
        RECOMMENDED_BATCH_SIZE = 128
    else:
        RECOMMENDED_BATCH_SIZE = 64
    print(f"\nRecommended batch size: {RECOMMENDED_BATCH_SIZE}")
else:
    print("\nNO GPU DETECTED")
    RECOMMENDED_BATCH_SIZE = 32

In [ ]:
import os

REPO_URL = "https://github.com/MaeTobiGeri/Geomar_AI_Oxygen.git"
REPO_NAME = "Geomar_AI_Oxygen"

if os.path.exists(REPO_NAME):
    print(f"Repository exists. Pulling latest changes...")
    !cd {REPO_NAME} && git pull
else:
    print(f"Cloning from {REPO_URL}...")
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
print(f"\nWorking directory: {os.getcwd()}")

In [ ]:
!pip install -q pytorch-forecasting==1.7.0
!pip install -q 'lightning>=2.0.0,<2.7.0'
!pip install -q 'wetterdienst>=0.90.0'
!pip install -q 'plotly>=5.0.0'
!pip install -q 'optuna>=3.0.0'

print("\n" + "="*80)
print("VERIFYING INSTALLATION")
print("="*80 + "\n")

import pytorch_forecasting
import lightning.pytorch as pl
import optuna

print(f"pytorch-forecasting: {pytorch_forecasting.__version__}")
print(f"lightning: {pl.__version__}")
print(f"optuna: {optuna.__version__}")

In [ ]:
from google.colab import drive

MOUNT_DRIVE = True

if MOUNT_DRIVE:
    drive.mount('/content/drive')
    CHECKPOINT_DIR = '/content/drive/MyDrive/Geomar_Checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"\nCheckpoints will be saved to Google Drive: {CHECKPOINT_DIR}")
else:
    CHECKPOINT_DIR = 'models/hypoxia_tft'
    print(f"\nCheckpoints will be saved locally: {CHECKPOINT_DIR}")

## 2. Verify Data and Pipeline

In [ ]:
!ls -lh Documentation/data/

from src import data_ingestion

df_ocean = data_ingestion._load_ocean_data()
print(f"\nOcean data loaded: {len(df_ocean)} rows")
print(f"Date range: {df_ocean['Date'].min()} to {df_ocean['Date'].max()}")

In [ ]:
!pip install -q pytest
!python -m pytest tests/ -v --tb=short

## 3. Training Options

### Option A: Quick Test (5 epochs)

In [ ]:
!python train.py \
    --max-epochs 5 \
    --batch-size 32 \
    --checkpoint-path "{CHECKPOINT_DIR}" \
    --patience 2

### Option B: Full Training

In [ ]:
!python train.py \
    --max-epochs 100 \
    --batch-size {RECOMMENDED_BATCH_SIZE} \
    --checkpoint-path "{CHECKPOINT_DIR}" \
    --patience 3

### Option C: Hyperparameter Tuning

In [ ]:
N_TRIALS = 20

!python tune_hyperparameters.py \
    --n-trials {N_TRIALS} \
    --output tuned_hyperparameters.json

import json
with open('tuned_hyperparameters.json') as f:
    tuned = json.load(f)

print("\n" + "="*80)
print("BEST HYPERPARAMETERS")
print("="*80)
for key, value in tuned['hyperparameters'].items():
    print(f"  {key}: {value}")
print(f"\nBest val_loss: {tuned['val_loss']:.4f}")

In [ ]:
!python train.py \
    --load-hyperparameters tuned_hyperparameters.json \
    --checkpoint-path "{CHECKPOINT_DIR}" \
    --max-epochs 100 \
    --patience 3

## 4. Advanced Model Analysis

### Full Dataset Predictions

In [ ]:
!python generate_full_predictions.py \
    --checkpoint-path "{CHECKPOINT_DIR}/best_model.ckpt" \
    --output-dir "{CHECKPOINT_DIR}/full_predictions" \
    --encoder-length 8 \
    --decoder-length 4

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import json

prediction_dir = Path(f"{CHECKPOINT_DIR}/full_predictions")
plot_path = prediction_dir / "full_dataset_predictions.png"

if plot_path.exists():
    print("="*80)
    print("FULL DATASET PREDICTIONS")
    print("="*80)
    
    display(Image(filename=str(plot_path)))
    
    metrics_path = prediction_dir / "metrics.json"
    if metrics_path.exists():
        with open(metrics_path) as f:
            metrics = json.load(f)
        
        print("\n" + "="*80)
        print("UNCERTAINTY QUANTIFICATION METRICS")
        print("="*80)
        
        std_metrics = metrics['standard_metrics']
        print(f"\nStandard Metrics:")
        print(f"  MAE:  {std_metrics['mae']:.2f} µmol/L")
        print(f"  RMSE: {std_metrics['rmse']:.2f} µmol/L")
        print(f"  MAPE: {std_metrics['mape']:.2f}%")
        
        print(f"\nUncertainty:")
        print(f"  PICP (P10-P90): {metrics['picp_p10_p90']:.2%}")
        print(f"  MPIW: {metrics['mpiw_raw']:.2f} µmol/L")
        
        print(f"\nStratified by Hypoxia Tier:")
        print(f"  {'Tier':<20} {'N':<8} {'MAE':<10} {'PICP':<10}")
        print(f"  {'-'*48}")
        
        for tier_name, tier_metrics in metrics['stratified'].items():
            n = tier_metrics['n_samples']
            mae = tier_metrics.get('mae', float('nan'))
            picp = tier_metrics.get('picp', float('nan'))
            
            tier_display = tier_name.replace('_', ' ').title()
            
            if n > 0:
                print(f"  {tier_display:<20} {n:<8} {mae:<10.2f} {picp:<10.2%}")
else:
    print("Full dataset predictions not found.")

### Autoencoder Feature Selection

In [ ]:
from src import data_ingestion, pipeline, labeling, features, feature_selection

df_combined = data_ingestion.load_and_clean_boknis_data()
df_weekly = pipeline.prepare_weekly_series(df_combined)
df_25m = labeling.select_target_series(df_weekly)
df_features = features.engineer_features(df_25m, df_weekly)
df_labeled = labeling.label_hypoxia_risk(df_features)
df_labeled = df_labeled.dropna()

feature_cols = [col for col in df_labeled.columns
                if col not in ['Date', 'O2_umol_L', 'Time_Idx', 'sample_weight',
                              'oxygen_deficit', 'month_sin', 'month_cos', 'Depth_m']]

results = feature_selection.hybrid_feature_selection(
    features_df=df_labeled,
    target_column='O2_umol_L',
    feature_columns=feature_cols,
    correlation_threshold=0.3,
    n_autoencoder_features=15,
    hidden_dim=8,
    epochs=100,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print("\n" + "="*80)
print("SELECTED FEATURES")
print("="*80)
for i, feat in enumerate(results['selected_features'], 1):
    score = results['autoencoder_scores'][feat]
    print(f"  {i:2d}. {feat:<30} (score: {score:.4f})")

## 5. Interactive Dashboard

In [ ]:
!pip install -q streamlit
!wget -q -O cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb

In [ ]:
import subprocess
import time
import re
from IPython.display import clear_output

with open('app.py', 'r') as f:
    content = f.read()

content = content.replace(
    'DEFAULT_CHECKPOINT = "models/hypoxia_tft/best_model.ckpt"',
    f'DEFAULT_CHECKPOINT = "{CHECKPOINT_DIR}/best_model.ckpt"'
)
content = content.replace(
    'predictions_path = Path("outputs/full_predictions/predictions.csv")',
    f'predictions_path = Path("{CHECKPOINT_DIR}/full_predictions/predictions.csv")'
)
content = content.replace(
    'interactive_plot_path = Path("outputs/full_predictions/full_dataset_predictions_interactive.html")',
    f'interactive_plot_path = Path("{CHECKPOINT_DIR}/full_predictions/full_dataset_predictions_interactive.html")'
)

with open('app_colab.py', 'w') as f:
    f.write(content)

streamlit_process = subprocess.Popen(
    ['streamlit', 'run', 'app_colab.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false',
     '--server.enableXsrfProtection', 'false'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(15)

tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8501'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

url = None
for _ in range(30):
    line = tunnel_process.stdout.readline()
    if not line:
        time.sleep(0.5)
        continue
    
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = match.group(0)
            break

if url:
    clear_output(wait=True)
    print("=" * 80)
    print("DASHBOARD IS LIVE")
    print("=" * 80)
    print(f"\nAccess dashboard at: {url}")
    print("\nKeep this cell running to keep dashboard alive")
    print("=" * 80)
    
    try:
        tunnel_process.wait()
    except KeyboardInterrupt:
        streamlit_process.terminate()
        tunnel_process.terminate()
else:
    print("Could not extract tunnel URL. Check output for trycloudflare.com")

## 6. View Results

In [ ]:
import json
from pathlib import Path

metadata_path = Path(CHECKPOINT_DIR) / "training_metadata.json"

if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)

    print("="*80)
    print("TRAINING METADATA")
    print("="*80)
    print(f"\nTraining Date: {metadata['training_date']}")

    print("\nHyperparameters:")
    for key, value in metadata['hyperparameters'].items():
        print(f"  {key}: {value}")

    print(f"\nDataset: {metadata['dataset_info']['total_samples']} samples")
    print(f"  Train: {metadata['dataset_info']['train_samples']}")
    print(f"  Val: {metadata['dataset_info']['val_samples']}")
else:
    print("No training metadata found.")